# RAG 评估指标 · 02 — Retrieval Ranking Metrics

**本 Notebook 目标：** 从零实现检索排序评估指标，理解 top-k 与排序质量对 Retriever 的影响。

## 学习目标

- 理解 Hit Rate@k、Precision@k、Recall@k、MRR@k、NDCG@k 的数学定义
- 区分「是否命中」「噪声比例」「漏召回」「排序质量」四类评估维度
- 对比不同 top-k 配置下的指标变化

> top-k 里是否命中？相关文档排得靠不靠前？

## 五大指标一览

```mermaid
%%{init: {'flowchart': {'curve': 'linear', 'rankSpacing': 45}}}%%
flowchart TB
    RET[检索排序列表 Top-K] --> H[HitRate@k 有没有命中]
    RET --> P[Precision@k 噪声比例]
    RET --> R[Recall@k 漏召回比例]
    RET --> M[MRR@k 首个相关排第几]
    RET --> N[NDCG@k 高相关是否靠前]
```

| 指标 | 回答的问题 |
|------|------------|
| **HitRate@k** | top-k 里是否至少有一个相关文档？ |
| **Precision@k** | top-k 里相关文档占比多少？ |
| **Recall@k** | 全部相关文档中找回了多少？ |
| **MRR@k** | 第一个相关文档排第几？ |
| **NDCG@k** | 高相关文档是否排在前面？ |

---
# Chapter 3 · Ranking Metrics

## MRR 定义与直觉

**公式**

$$\text{MRR} = \frac{1}{|Q|} \sum_{i=1}^{|Q|} \frac{1}{\text{rank}_i}$$

| 符号 | 含义 |
|------|------|
| `|Q|` | 查询集合中的 query 总数 |
| `rank_i` | 第 i 个 query 中，**第一个相关 chunk 的排名位置** |
| `1/rank_i` | 排名的倒数（Reciprocal Rank） |

**直觉解释**

- 第 1 名就是相关的：RR = 1/1 = **1.0**（满分）
- 第 2 名才是相关的：RR = 1/2 = **0.5**
- 第 5 名才是相关的：RR = 1/5 = **0.2**（很差）
- 没有相关 chunk：RR = **0**

> MRR 特别适合"用户通常只看第一个结果"的场景（搜索引擎、问答系统）

## MRR 排名 → 得分

```mermaid
%%{init: {'flowchart': {'curve': 'linear', 'rankSpacing': 45}}}%%
flowchart TB
    R1[第1名相关] --> RR1[RR = 1.0]
    R2[第2名相关] --> RR2[RR = 0.5]
    R3[第3名相关] --> RR3[RR = 0.33]
    R0[top-k 无相关] --> RR0[RR = 0]
    RR1 --> MRR[MRR = 各 query RR 均值]
    RR2 --> MRR
    RR3 --> MRR
    RR0 --> MRR
```

## MRR 计算示例

| Query | 检索结果排名 | 第一个相关排名 | RR |
|-------|------------|-------------|-----|
| Q1: "什么是 RAG？" | [相关, 不相关, 相关] | 第 1 名 | 1/1 = 1.0 |
| Q2: "如何优化 chunking？" | [不相关, 相关, 不相关] | 第 2 名 | 1/2 = 0.5 |
| Q3: "向量数据库选型？" | [不相关, 不相关, 相关] | 第 3 名 | 1/3 = 0.33 |

$$\text{MRR} = \frac{1}{3}(1.0 + 0.5 + 0.33) \approx 0.61$$

> 经验阈值：MRR > 0.7 良好；MRR < 0.5 需要改进排序策略

## MRR@k 计算流程

```mermaid
%%{init: {'flowchart': {'curve': 'linear', 'rankSpacing': 45}}}%%
flowchart TB
    L[retrieved 列表] --> SCAN[从 rank=1 扫描 top-k]
    SCAN --> HIT{遇到相关 doc?}
    HIT -->|是| RR[RR = 1/rank]
    HIT -->|否继续| SCAN
    HIT -->|扫完无| Z[RR = 0]
    RR --> AVG[对所有 query 求均值]
    Z --> AVG
```

## MRR vs NDCG 对比

| 维度 | MRR | NDCG |
|------|-----|------|
| **全称** | Mean Reciprocal Rank | Normalized Discounted Cumulative Gain |
| **相关性** | 二值（相关/不相关） | 多级（0/1/2/3 分） |
| **考虑范围** | 只看第一个相关结果 | 看全部 K 个结果 |
| **排名敏感** | 是 | 是（对数折扣） |
| **适合场景** | 问答系统、首位匹配 | 搜索引擎、推荐系统 |
| **RAG 使用** | 常用 | 需要多级标注，成本高 |

**何时用 MRR？** 知识库 chunk 的相关性是二值的；用户主要依赖第一个检索结果。

**何时考虑 NDCG？** 有精细的相关性标注（高度相关/部分相关/不相关）；需要评估 Top-10 以上的结果质量。

## MRR vs NDCG 关注点

```mermaid
%%{init: {'flowchart': {'curve': 'linear', 'rankSpacing': 45}}}%%
flowchart TB
    RET[Top-K 排序列表]
    RET --> MRR[MRR 只看第一个相关]
    RET --> NDCG[NDCG 看全部位置 graded gain]
    NDCG --> DCG[DCG 折损累加]
    DCG --> NORM[除以 IDCG 归一化]
```

## HitRate / Precision / Recall @k

```mermaid
%%{init: {'flowchart': {'curve': 'linear', 'rankSpacing': 45}}}%%
flowchart TB
    TK[取 retrieved 前 k 条] --> H{至少1个相关?}
    H -->|是| HR[HitRate = 1]
    H -->|否| HR0[HitRate = 0]
    TK --> PC[hits / k = Precision@k]
    TK --> RC[hits / GT总数 = Recall@k]
```

## NDCG@k 计算流程

```mermaid
%%{init: {'flowchart': {'curve': 'linear', 'rankSpacing': 45}}}%%
flowchart TB
    G[gain = relevance grade] --> DCG[DCG = sum gain/log2 rank+1]
    IDEAL[理想排序 IDCG] --> NORM[NDCG = DCG / IDCG]
    DCG --> NORM
```

排名越靠后，gain 折损越大 → 鼓励把高相关 doc 排在前面。

## Top-k 配置对比示例（预期输出）

| k | HitRate | Precision | Recall | MRR | NDCG |
|---|---------|-----------|--------|-----|------|
| 1 | 0.25 | 0.25 | 0.08 | 0.25 | 0.25 |
| 3 | 0.75 | 0.42 | 0.54 | 0.46 | 0.46 |
| 5 | 1.00 | 0.45 | 1.00 | 0.52 | 0.64 |

**诊断**

- k 从 1 到 5，Recall 明显提升：说明 top-k 太小会漏召回
- Precision 没明显提升：说明扩大 k 会带来更多噪声
- NDCG 仍不高：说明相关文档虽然找到了，但排序还可以优化

## 增大 k 的指标变化

```mermaid
%%{init: {'flowchart': {'curve': 'linear', 'rankSpacing': 45}}}%%
flowchart TB
    KUP[增大 k] --> RUP[Recall 上升]
    KUP --> HUP[HitRate 上升]
    KUP --> PDOWN[Precision 可能下降或停滞]
    KUP --> MUP[MRR/NDCG 缓慢上升]
```

---
# Part 1 · 评估数据集（代码实战）

4 条模拟检索结果，每条包含 `retrieved`（排序列表）和 `relevance`（graded relevance：0=不相关，1/2/3=弱/中/强相关）。

In [4]:
from math import log2

EVAL_CASES = [
    {
        "query": "什么是 RAG 技术？",
        "retrieved": ["doc1", "doc8", "doc4", "doc2", "doc6"],
        "relevance": {"doc1": 3, "doc2": 2, "doc8": 1},
    },
    {
        "query": "如何减少 LLM 幻觉？",
        "retrieved": ["doc3", "doc5", "doc1", "doc7", "doc8"],
        "relevance": {"doc1": 3, "doc5": 2},
    },
    {
        "query": "Reranker 有什么作用？",
        "retrieved": ["doc7", "doc2", "doc4", "doc3", "doc8"],
        "relevance": {"doc4": 3, "doc3": 1},
    },
    {
        "query": "什么是 Hybrid Search？",
        "retrieved": ["doc5", "doc8", "doc1", "doc2", "doc7"],
        "relevance": {"doc7": 3, "doc2": 1},
    },
]


def relevant_ids(case: dict) -> set:
    """把 graded relevance 转成 binary relevance 集合。"""
    return {doc_id for doc_id, grade in case["relevance"].items() if grade > 0}

---
# Part 2 · 核心指标函数

实现 Hit Rate@k、Precision@k、Recall@k、MRR@k、NDCG@k 及汇总函数。

## 代码实现映射

```mermaid
%%{init: {'flowchart': {'curve': 'linear', 'rankSpacing': 45}}}%%
flowchart TB
    CASE[EVAL_CASES] --> H[hit_rate_at_k]
    CASE --> P[precision_at_k]
    CASE --> R[recall_at_k]
    CASE --> M[reciprocal_rank_at_k]
    CASE --> N[ndcg_at_k]
    H --> SUM[summarize_at_k]
    P --> SUM
    R --> SUM
    M --> SUM
    N --> SUM
```

In [6]:
def hit_rate_at_k(case: dict, k: int) -> float:
    """Hit Rate@k = top-k 中是否至少命中 1 个相关文档。"""
    top_k = case["retrieved"][:k]
    return 1.0 if set(top_k) & relevant_ids(case) else 0.0


def precision_at_k(case: dict, k: int) -> float:
    """Precision@k = top-k 中相关文档数 / k。"""
    top_k = case["retrieved"][:k]
    hits = sum(1 for doc_id in top_k if doc_id in relevant_ids(case))
    return hits / k if k else 0.0


def recall_at_k(case: dict, k: int) -> float:
    """Recall@k = top-k 中相关文档数 / 全部相关文档数。"""
    rel_ids = relevant_ids(case)
    if not rel_ids:
        return 0.0
    top_k = case["retrieved"][:k]
    hits = sum(1 for doc_id in top_k if doc_id in rel_ids)
    return hits / len(rel_ids)


def reciprocal_rank_at_k(case: dict, k: int) -> float:
    """RR@k = 第一个相关文档排名的倒数；top-k 没命中则为 0。"""
    rel_ids = relevant_ids(case)
    for rank, doc_id in enumerate(case["retrieved"][:k], start=1):
        if doc_id in rel_ids:
            return 1.0 / rank
    return 0.0


def dcg_at_k(case: dict, k: int) -> float:
    """DCG@k = graded relevance 按排名折损后的收益。"""
    score = 0.0
    for rank, doc_id in enumerate(case["retrieved"][:k], start=1):
        gain = case["relevance"].get(doc_id, 0)
        score += gain / log2(rank + 1)
    return score


def ndcg_at_k(case: dict, k: int) -> float:
    """NDCG@k = 当前排序 DCG / 理想排序 IDCG。"""
    ideal_grades = sorted(case["relevance"].values(), reverse=True)
    idcg = sum(grade / log2(rank + 1) for rank, grade in enumerate(ideal_grades[:k], start=1))
    if idcg == 0:
        return 0.0
    return dcg_at_k(case, k) / idcg


def mean(values: list[float]) -> float:
    return sum(values) / len(values) if values else 0.0


def summarize_at_k(cases: list[dict], k: int) -> dict:
    return {
        "hit_rate": mean([hit_rate_at_k(case, k) for case in cases]),
        "precision": mean([precision_at_k(case, k) for case in cases]),
        "recall": mean([recall_at_k(case, k) for case in cases]),
        "mrr": mean([reciprocal_rank_at_k(case, k) for case in cases]),
        "ndcg": mean([ndcg_at_k(case, k) for case in cases]),
    }

---
# Part 3 · 逐条查询明细（@k=3）

对 4 条查询分别输出 Hit、Precision、Recall、RR、NDCG。

In [7]:
def print_case_details(k: int = 3) -> None:
    print("=" * 80)
    print(f"Retrieval Ranking Metrics 明细（@{k}）")
    print("=" * 80)
    print(f"{'查询':<22} {'Top-k':<28} {'Hit':>5} {'P@k':>7} {'R@k':>7} {'RR@k':>7} {'NDCG':>7}")
    print("-" * 80)

    for case in EVAL_CASES:
        top_k = ",".join(case["retrieved"][:k])
        print(
            f"{case['query'][:20]:<22} {top_k:<28} "
            f"{hit_rate_at_k(case, k):>5.0f} "
            f"{precision_at_k(case, k):>7.2f} "
            f"{recall_at_k(case, k):>7.2f} "
            f"{reciprocal_rank_at_k(case, k):>7.2f} "
            f"{ndcg_at_k(case, k):>7.2f}"
        )


print_case_details(k=3)

Retrieval Ranking Metrics 明细（@3）
查询                     Top-k                          Hit     P@k     R@k    RR@k    NDCG
--------------------------------------------------------------------------------
什么是 RAG 技术？            doc1,doc8,doc4                   1    0.67    0.67    1.00    0.76
如何减少 LLM 幻觉？           doc3,doc5,doc1                   1    0.67    1.00    0.50    0.65
Reranker 有什么作用？        doc7,doc2,doc4                   1    0.33    0.50    0.33    0.41
什么是 Hybrid Search？     doc5,doc8,doc1                   0    0.00    0.00    0.00    0.00


---
# Part 4 · Top-k 配置对比

对比 k=1 / 3 / 5 下的平均指标，观察 top-k 变化对各维度的影响。

In [8]:
def print_k_sweep() -> None:
    print("\n" + "=" * 80)
    print("Top-k 配置对比")
    print("=" * 80)
    print(f"{'k':>3} {'HitRate':>9} {'Precision':>10} {'Recall':>8} {'MRR':>8} {'NDCG':>8}")
    print("-" * 55)

    for k in [1, 3, 5]:
        summary = summarize_at_k(EVAL_CASES, k)
        print(
            f"{k:>3} "
            f"{summary['hit_rate']:>9.2f} "
            f"{summary['precision']:>10.2f} "
            f"{summary['recall']:>8.2f} "
            f"{summary['mrr']:>8.2f} "
            f"{summary['ndcg']:>8.2f}"
        )

    print("\n解读：")
    print("  HitRate@k 适合判断 top-k 是否至少能找到答案线索。")
    print("  Precision@k 适合观察检索噪声。")
    print("  Recall@k 适合观察漏召回。")
    print("  MRR@k / NDCG@k 适合观察排序质量。")
    print("=" * 80)


print_k_sweep()


Top-k 配置对比
  k   HitRate  Precision   Recall      MRR     NDCG
-------------------------------------------------------
  1      0.25       0.25     0.08     0.25     0.25
  3      0.75       0.42     0.54     0.46     0.46
  5      1.00       0.45     1.00     0.52     0.64

解读：
  HitRate@k 适合判断 top-k 是否至少能找到答案线索。
  Precision@k 适合观察检索噪声。
  Recall@k 适合观察漏召回。
  MRR@k / NDCG@k 适合观察排序质量。
